# WM-811K 2단계: lot 누출 방지 CNN 학습

1단계에서 Google Drive에 저장한 64×64 배열을 사용합니다. **런타임 → 런타임 유형 변경 → T4 GPU**를 선택한 뒤 위에서부터 실행하세요.

- 입력 채널: 웨이퍼 영역과 불량 다이 영역 2개
- 분할: lot 기준 약 71/14/14%, 세 세트 간 lot 중복 0
- 불균형 대응: class-balanced focal loss
- 증강: 90도 회전과 좌우·상하 반전
- 선택 지표: validation macro-F1

In [ ]:
!pip install -q pandas==2.2.3 scikit-learn matplotlib

from google.colab import drive, files
from pathlib import Path
import json, shutil, subprocess, sys
import pandas as pd
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'T4 GPU 런타임을 선택한 뒤 다시 실행하세요.'
print('GPU:', torch.cuda.get_device_name(0))

DRIVE_DATA_DIR = Path('/content/drive/MyDrive/AI_project/WM811K_processed')
LOCAL_DATA_DIR = Path('/content/wm811k_processed')
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
for name in ('wafer_maps_64.npy', 'labels.npy'):
    source = DRIVE_DATA_DIR / name
    assert source.is_file(), f'1단계 출력 파일이 없습니다: {source}'
    destination = LOCAL_DATA_DIR / name
    if not destination.is_file() or destination.stat().st_size != source.stat().st_size:
        print('로컬 SSD로 복사:', name)
        shutil.copy2(source, destination)
print('학습 데이터 준비 완료:', LOCAL_DATA_DIR)

## 학습 코드와 고정 분할 업로드

로컬 프로젝트에서 아래 두 파일을 동시에 선택하세요.

- `코드/wm811k/train_wm811k_cnn.py`
- `결과물/wm811k/split_results/split_assignments.csv`

In [ ]:
uploaded = {}
print('1/2: train_wm811k_cnn.py를 선택하세요.')
uploaded.update(files.upload())
print('2/2: split_assignments.csv를 선택하세요.')
uploaded.update(files.upload())

def uploaded_bytes(prefix, suffix):
    matches = [name for name in uploaded if name.startswith(prefix) and name.endswith(suffix)]
    assert matches, f'선택하지 않은 파일: {prefix}{suffix}'
    return uploaded[matches[-1]]

TRAIN_SCRIPT = Path('/content/train_wm811k_cnn.py')
ASSIGNMENTS = Path('/content/split_assignments.csv')
TRAIN_SCRIPT.write_bytes(uploaded_bytes('train_wm811k_cnn', '.py'))
ASSIGNMENTS.write_bytes(uploaded_bytes('split_assignments', '.csv'))
print('업로드 완료')

## T4 GPU 학습

최대 20 epoch이며 validation macro-F1이 4 epoch 동안 개선되지 않으면 조기 종료합니다.

In [ ]:
OUTPUT_DIR = Path('/content/drive/MyDrive/AI_project/WM811K_training/focal_baseline')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, '-u', str(TRAIN_SCRIPT),
    '--maps', str(LOCAL_DATA_DIR / 'wafer_maps_64.npy'),
    '--labels', str(LOCAL_DATA_DIR / 'labels.npy'),
    '--assignments', str(ASSIGNMENTS),
    '--output-dir', str(OUTPUT_DIR),
    '--epochs', '20',
    '--batch-size', '256',
    '--num-workers', '2',
    '--loss', 'focal',
    '--seed', '42',
]
subprocess.run(command, check=True)
print('CNN 학습 완료:', OUTPUT_DIR)

## 결과 확인 및 전달용 ZIP 다운로드

모델과 검증 결과를 ZIP으로 내려받습니다. Drive의 전체 학습 결과는 그대로 유지하세요.

In [ ]:
summary = json.loads((OUTPUT_DIR / 'run_summary.json').read_text(encoding='utf-8'))
print(json.dumps(summary, ensure_ascii=False, indent=2))
display(pd.read_csv(OUTPUT_DIR / 'test_classification_report.csv'))

receipt_dir = Path('/content/wm811k_cnn_receipt')
if receipt_dir.exists():
    shutil.rmtree(receipt_dir)
receipt_dir.mkdir()
for name in (
    'best_model.pt', 'run_summary.json', 'training_history.csv',
    'training_curves.png', 'test_classification_report.csv',
    'test_confusion_matrix.csv', 'test_confusion_matrix.png',
    'test_predictions.csv',
):
    source = OUTPUT_DIR / name
    assert source.is_file(), f'결과 파일이 없습니다: {source}'
    shutil.copy2(source, receipt_dir / name)
archive = shutil.make_archive('/content/wm811k_cnn_results', 'zip', receipt_dir)
print('전달용 결과:', archive)
files.download(archive)

다운로드한 `wm811k_cnn_results.zip`을 압축 해제하지 말고 로컬 프로젝트의 `결과물/wm811k/colab_가져오기/`에 넣은 뒤 Codex에 알려주세요.